<a href="https://colab.research.google.com/github/SmallMGarden/scientist_girl_in_green/blob/main/%D0%9A%D0%BE%D0%BF%D0%B8%D1%8F_%D0%B1%D0%BB%D0%BE%D0%BA%D0%BD%D0%BE%D1%82%D0%B0_%22fine_tuning_hw_ipynb%22.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#### Домашнее задание

**Датасет:** [`ag_news`](https://huggingface.co/datasets/fancyzhx/ag_news) — классификация новостей по 4-м категориям (World, Sports, Business, Sci/Tech)

**Техническое задание:**

1.  Загрузите датасет `ag_news`
2.  Выберите модель для дообучения (например, `distilbert-base-uncased` или `bert-base-uncased`), `num_labels=4`
3.  Токенизируйте данные (`max_length=128`)
4.  Настройте `TrainingArguments`:
    *   `learning_rate = 2e-5`
    *   `per_device_train_batch_size = 16`
    *   `num_train_epochs = 3`
    *   `eval_strategy = "epoch"`
    *   `save_strategy = "epoch"`
    *   `load_best_model_at_end = True`
    *   `metric_for_best_model = "accuracy"`
5.  Обучите модель с помощью `Trainer`. Для метрик используйте `evaluate.load("accuracy")`
6.  Выведите accuracy на тестовой выборке
7.  Сохраните модель в папку `./ag_news_model`
8.  Протестируйте модель на трех новых новостях (вписать вручную), используя `pipeline`. Выведите предсказанный класс и уверенность модели

In [16]:
!pip install transformers datasets evaluate

In [ ]:
from datasets import load_dataset

dataset = load_dataset("ag_news")

dataset
#импортируем датасетик

DatasetDict({
    train: Dataset({
        features: ['text', 'label'],
        num_rows: 120000
    })
    test: Dataset({
        features: ['text', 'label'],
        num_rows: 7600
    })
})

In [ ]:
#модель для дообучения: bert-base-uncased
from transformers import AutoTokenizer

model_name = "bert-base-uncased"

tokenizer = AutoTokenizer.from_pretrained(model_name)

config.json:   0%|          | 0.00/570 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/48.0 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

In [ ]:
def tokenize_function(example):
    return tokenizer(
        example["text"],
        padding="max_length",
        truncation=True,
        max_length=128
    )
tokenized_datasets = dataset.map(tokenize_function, batched=True)#применяем токенизацию ко всему датасету
tokenized_datasets = tokenized_datasets.remove_columns(["text"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")#удаляем исходный текст

Map:   0%|          | 0/120000 [00:00<?, ? examples/s]

In [ ]:
from transformers import AutoModelForSequenceClassification

model = AutoModelForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=4
)
#добявляем условие num_labels=4

model.safetensors:   0%|          | 0.00/440M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

BertForSequenceClassification LOAD REPORT from: bert-base-uncased
Key                                        | Status     | 
-------------------------------------------+------------+-
cls.predictions.bias                       | UNEXPECTED | 
cls.seq_relationship.weight                | UNEXPECTED | 
cls.predictions.transform.dense.bias       | UNEXPECTED | 
cls.predictions.transform.LayerNorm.bias   | UNEXPECTED | 
cls.seq_relationship.bias                  | UNEXPECTED | 
cls.predictions.transform.dense.weight     | UNEXPECTED | 
cls.predictions.transform.LayerNorm.weight | UNEXPECTED | 
classifier.bias                            | MISSING    | 
classifier.weight                          | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


In [ ]:
import evaluate

accuracy = evaluate.load("accuracy")

In [ ]:
import numpy as np

def compute_metrics(eval_pred):
    logits, labels = eval_pred  # logits — предсказания модели, labels — реальные классы
    predictions = np.argmax(logits, axis=1)  # выбираем класс с максимальной вероятностью
    return accuracy.compute(predictions=predictions, references=labels)  # считаем accuracy

In [ ]:
#настраиваем параметры для обучения:
#TrainingArguments:
#learning_rate = 2e-5
#per_device_train_batch_size = 16
#num_train_epochs = 3
#eval_strategy = "epoch"
#save_strategy = "epoch"
#load_best_model_at_end = True
#metric_for_best_model = "accuracy"

from transformers import TrainingArguments

training_args = TrainingArguments(
    output_dir="./lisanasobake",
    learning_rate=2e-5,
    per_device_train_batch_size=16,
    num_train_epochs=3,
    eval_strategy="epoch",
    save_strategy="epoch",
    load_best_model_at_end=True,
    metric_for_best_model="accuracy"
)

In [15]:
# создаем Trainer

from transformers import Trainer

trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=tokenized_datasets["train"],
    eval_dataset=tokenized_datasets["test"],
    compute_metrics=compute_metrics  # функция для подсчета accuracy
)

trainer.train()

Epoch,Training Loss,Validation Loss


Epoch,Training Loss,Validation Loss,Accuracy
1,0.199747,0.186240,0.943947
2,0.128916,0.193368,0.948684


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [17]:
# оценка модели на тестовой выборке
eval_results = trainer.evaluate(tokenized_datasets["test"])
print("Accuracy на тесте:", eval_results["eval_accuracy"])

Epoch,Training Loss,Validation Loss,Accuracy
1,0.199747,0.186240,0.943947
2,0.090006,0.226201,0.948684


Accuracy на тесте: 0.9486842105263158


In [18]:
trainer.save_model("./ag_news_model")

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

In [20]:
# создаём pipeline для классификации новостей

from transformers import pipeline

classifier = pipeline(
    "text-classification",
    model="./ag_news_model",
    tokenizer="./ag_news_model",
    return_all_scores=True
)

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [24]:
from transformers import pipeline

# создаём pipeline
classifier = pipeline(
    "text-classification",
    model="./ag_news_model",
    tokenizer="./ag_news_model"
)

new_texts = [
    "Yesterday, a dog on a fox was found in HSE",
    "HSE is discussing the issue of recently stolen bicycles",
    "HSE students said they have nightmares about linguistics"
]

# словарь для красивых меток
label_map = {
    "LABEL_0": "World",
    "LABEL_1": "Sports",
    "LABEL_2": "Business",
    "LABEL_3": "Sci/Tech"
}

# предсказания
predictions = classifier(new_texts)

# выводим результаты
for text, pred in zip(new_texts, predictions):
    print("\nТекст новости:", text)
    print(f"Предсказанный класс: {label_map[pred['label']]}")
    print(f"Уверенность: {pred['score']:.3f}")

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]


Текст новости: Yesterday, a dog on a fox was found in HSE
Предсказанный класс: Sports
Уверенность: 0.548

Текст новости: HSE is discussing the issue of recently stolen bicycles
Предсказанный класс: Sci/Tech
Уверенность: 0.682

Текст новости: HSE students said they have nightmares about linguistics
Предсказанный класс: Sci/Tech
Уверенность: 0.822
